# Retail Customer Intelligence Platform
Advanced customer analytics using Python, SQL and Power BI.

In [ ]:
import pandas as pd
import numpy as np
orders=pd.read_csv('../data/raw/orders.csv',parse_dates=['order_date'])
customers=pd.read_csv('../data/raw/customers.csv',parse_dates=['signup_date'])
products=pd.read_csv('../data/raw/products.csv')
valid=orders[orders.order_status.isin(['Delivered','Returned'])].copy()
valid.head()

## Executive KPIs

In [ ]:
kpis=pd.Series({'Revenue':valid.net_sales_inr.sum(),'Orders':valid.order_id.nunique(),'Customers':valid.customer_id.nunique(),'AOV':valid.net_sales_inr.sum()/valid.order_id.nunique(),'Contribution Margin %':valid.contribution_profit_inr.sum()/valid.net_sales_inr.sum()})
kpis

## RFM Segmentation

In [ ]:
snapshot=pd.Timestamp('2025-12-31')
rfm=valid.groupby('customer_id').agg(recency_days=('order_date',lambda x:(snapshot-x.max()).days),frequency=('order_id','nunique'),monetary=('net_sales_inr','sum')).reset_index()
rfm['R']=pd.qcut(rfm.recency_days.rank(method='first'),5,labels=[5,4,3,2,1]).astype(int)
rfm['F']=pd.qcut(rfm.frequency.rank(method='first'),5,labels=[1,2,3,4,5]).astype(int)
rfm['M']=pd.qcut(rfm.monetary.rank(method='first'),5,labels=[1,2,3,4,5]).astype(int)
rfm['RFM_score']=rfm.R+rfm.F+rfm.M
rfm.sort_values('RFM_score',ascending=False).head(10)

## Product Profitability

In [ ]:
x=valid.merge(products[['product_id','product_name','category']],on='product_id')
product_profit=x.groupby(['category','product_name']).agg(revenue=('net_sales_inr','sum'),contribution=('contribution_profit_inr','sum'),orders=('order_id','nunique')).reset_index()
product_profit['margin_pct']=100*product_profit.contribution/product_profit.revenue
product_profit.sort_values('contribution',ascending=False).head(15)

## Cohort Retention

In [ ]:
x=valid[['customer_id','order_date']].copy()
x['order_month']=x.order_date.dt.to_period('M')
first=x.groupby('customer_id').order_date.transform('min').dt.to_period('M')
x['cohort_month']=first
x['cohort_age']=(x.order_month.astype(int)-x.cohort_month.astype(int))
cohort=x.groupby(['cohort_month','cohort_age']).customer_id.nunique().reset_index()
cohort.head()

## Business Decision Layer
Prioritize retention spend on customers with high CLV and high churn probability; investigate high-return/low-margin products; scale acquisition channels that generate strong contribution per customer.